# Epic 5 -- Active Domain Adaptation Overview

Epic 4 introduced DANN for unsupervised domain adaptation -- the encoder learns
domain-invariant features using a gradient reversal layer and **no** target labels.
This works well in many cases, but in practice the target-domain performance still
lags behind a fully-supervised model.  The reason is simple: without any labeled
target data the model has no direct signal about what the correct segmentation looks
like on the target domain.

**Active Domain Adaptation** closes this gap by strategically selecting a small
number of target-domain samples for human labeling.  Instead of labeling
everything (expensive) or nothing (DANN only), we pick the samples that will
improve the model the most.

## The labeling bottleneck

We have 4 manufacturing sites producing X-ray images:

| Site | Images | Labels | Role |
|------|--------|--------|------|
| Warstein (source) | 2000+ | Full masks | Training source |
| Malaysia (target) | 1500+ | None | Unlabeled target |
| Mexico (target) | 800+ | None | Unlabeled target |
| China (target) | 600+ | None | Unlabeled target |

Labeling every target image would cost weeks of expert time.  Active learning
asks: **which 50--100 target images should we label to get the biggest
improvement?**

## What you will learn

1. The active domain adaptation loop
2. How it builds on Epic 4 DANN
3. The architecture: uncertainty, selection, labeling, retraining
4. Quick demo: create a DANN model with MC Dropout enabled

## Prerequisites

- Python 3.12, PyTorch, scikit-learn
- Install: `uv pip install -e ".[epic5]"`
- Familiarity with Epic 4 DANN (see `epic4_00_overview.ipynb`)

---
## 1. Architecture: The Active Domain Adaptation Loop

The system runs in iterative **rounds**.  Each round:

```
    +----------------------------------------------------------+
    |                     ROUND r                              |
    |                                                          |
    |  +--------------+    +----------------+    +-----------+ |
    |  | DANN Model   |--->| MC Dropout     |--->| Uncertainty| |
    |  | (current)    |    | T=20 passes    |    | Scores    | |
    |  +--------------+    +----------------+    +-----+-----+ |
    |                                                  |       |
    |                                                  v       |
    |  +--------------+    +----------------+    +-----------+ |
    |  | Feature      |--->| Coreset /      |--->| Diversity | |
    |  | Extractor    |    | k-Means        |    | Scores    | |
    |  +--------------+    +----------------+    +-----+-----+ |
    |                                                  |       |
    |                                                  v       |
    |                                          +-----------+   |
    |                                          | Combined  |   |
    |                                          | Selection |   |
    |                                          +-----+-----+   |
    |                                                |         |
    |                                                v         |
    |                                          +-----------+   |
    |                                          | Human     |   |
    |                                          | Labeling  |   |
    |                                          +-----+-----+   |
    |                                                |         |
    |                                                v         |
    |                                          +-----------+   |
    |                                          | Retrain   |   |
    |                                          | DANN +    |   |
    |                                          | labels    |   |
    |                                          +-----------+   |
    +----------------------------------------------------------+
                              |
                              v
                       Repeat until convergence
```

### Step-by-step

| Step | Module | What happens |
|------|--------|--------------|
| 1. Uncertainty | `udm_epic5.uncertainty.mc_dropout` | Run T stochastic forward passes, compute per-pixel entropy |
| 2. Diversity | `udm_epic5.selection.diversity` | Extract features, select diverse samples via coreset or k-means |
| 3. Combined | `udm_epic5.selection.combined` | Merge uncertainty + diversity into a single ranking |
| 4. Labeling | `udm_epic5.labeling.session` | Create a session folder, human annotates selected samples |
| 5. Retrain | `udm_epic5.active_training.train_active_dann` | Train DANN with source labels + new target labels |
| 6. Analyse | `udm_epic5.analysis.convergence` | Plot learning curves, check stopping criterion |

---
## 2. How It Builds on Epic 4 DANN

Epic 5 reuses the Epic 4 DANN model directly:

| Epic 4 component | How Epic 5 uses it |
|------------------|--------------------|
| `DANNModel` | Base model -- we add MC Dropout on top |
| `SharedEncoder` | Feature extractor for diversity selection |
| `GradientReversalLayer` | Still used during Active DANN retraining |
| `DomainBatchSampler` | Extended to include labeled target samples |
| `train_dann_from_yaml` | Wrapped by `train_active_dann_from_yaml` |

The key addition is the **selection loop** that wraps around the training:
instead of training once, we train, select, label, and retrain in rounds.

### Why not just label randomly?

Random selection wastes the labeling budget on easy or redundant samples.
Active selection focuses on:
- **Uncertain** samples: the model does not know the answer (high entropy)
- **Diverse** samples: they cover different parts of the target distribution

Combining both criteria gives the best improvement per label.

---
## 3. Quick Demo: Create a DANN Model with MC Dropout

MC Dropout (Monte Carlo Dropout) is the foundation of uncertainty estimation
in Epic 5.  The idea is simple: keep dropout **enabled** during inference and
run multiple forward passes.  The variation across passes tells us how
uncertain the model is.

Let's create a DANN model and enable MC Dropout.

In [ ]:
import torch
import numpy as np

from udm_epic4.models.dann import DANNModel
from udm_epic5.uncertainty.mc_dropout import enable_mc_dropout, mc_dropout_uncertainty, compute_entropy

In [ ]:
# Create a DANN model (same as Epic 4)
model = DANNModel(
    backbone="convnext_tiny",
    pretrained=False,
    in_chans=3,
    decoder_channels=[256, 128, 64, 32],
    domain_head_hidden=256,
)
print(f"Model created.  Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Enable MC Dropout: this switches dropout layers to training mode
# while keeping everything else in eval mode.
enable_mc_dropout(model)

# Verify: the model is in eval mode, but dropout layers are in train mode.
for name, module in model.named_modules():
    if 'drop' in name.lower():
        print(f"  {name}: training={module.training}")

In [ ]:
# Run T=5 stochastic forward passes on a dummy image
dummy_input = torch.randn(1, 3, 512, 512)
T = 5

with torch.no_grad():
    # mc_dropout_uncertainty returns a probability stack [T, B, 1, H, W]
    prob_stack = mc_dropout_uncertainty(model, dummy_input, T=T)

print(f"Probability stack shape: {prob_stack.shape}")  # [5, 1, 1, 512, 512]
print(f"Each pass gives a different prediction due to dropout stochasticity.")
print(f"Mean prediction range: [{prob_stack.mean():.3f}]")
print(f"Std across passes:     [{prob_stack.std(dim=0).mean():.4f}]")

In [ ]:
# Compute per-pixel entropy from the probability stack
entropy_map = compute_entropy(prob_stack)  # [B, H, W]
print(f"Entropy map shape: {entropy_map.shape}")
print(f"Entropy range: [{entropy_map.min():.4f}, {entropy_map.max():.4f}]")
print(f"\nHigh entropy = high uncertainty = the model disagrees across passes.")
print(f"These are the pixels (and images) we want humans to label.")

---
## 4. Project Structure

| Module | Description |
|--------|------------|
| `udm_epic5.uncertainty.mc_dropout` | `enable_mc_dropout`, `mc_dropout_uncertainty`, `compute_entropy` |
| `udm_epic5.selection.diversity` | `coreset_selection`, `clustering_selection` |
| `udm_epic5.selection.combined` | `combined_selection`, `export_selection_csv` |
| `udm_epic5.labeling.session` | `LabelingSession`, `create_labeling_session`, `load_labeled_samples` |
| `udm_epic5.active_training.train_active_dann` | `train_active_dann_from_yaml` |
| `udm_epic5.analysis.convergence` | `learning_curve`, `compare_strategies`, `stopping_criterion` |

## CLI overview

```bash
# Compute uncertainty scores
udm-epic5 uncertainty --config configs/epic5_active.yaml

# Select samples (uncertainty, diversity, or combined)
udm-epic5 select --strategy combined --budget 50

# Prepare a labeling session
udm-epic5 prepare-session --round 1

# Train Active DANN with labeled target samples
udm-epic5 train --config configs/epic5_active.yaml

# Analyse convergence across rounds
udm-epic5 analyze --output results/convergence.png
```

## Next steps

| Notebook | Topic |
|----------|------|
| `epic5_01_uncertainty.ipynb` | MC Dropout uncertainty quantification |
| `epic5_02_diversity.ipynb` | Diversity-based sample selection |
| `epic5_03_combined.ipynb` | Combined selection strategy |
| `epic5_04_labeling.ipynb` | Human-in-the-loop labeling |
| `epic5_05_active_dann.ipynb` | Active DANN training |
| `epic5_06_convergence.ipynb` | Multi-round convergence analysis |